# Individual motif assessment

Occupancy of each Mg²⁺ binding motif in the simulations ($p_\mathrm{sim}$) and as estimated from the cryo-EM map ($p_\mathrm{cryo}$). For each motif, the simulated map is mixed from the frames where the motif is occupied (P) and those where it is not (A), $f\,P + (1 - f)\,A$; $p_\mathrm{cryo}$ is the $f$ with the best correlation with the half-map average of EMD-42499, within 6 Å of the motif atoms (in 9CBU). Each simulation is blurred with its best B-factor for the core phosphate shell (as in 05). The "+ Cryo" estimates pool the 7ez0 and 7ez0_noion simulations of a method and force field. Families: rigid fit of the RNA ("global") and residue-by-residue fit ("local"), as in 05.

$p_\mathrm{sim}$: production window, λ₀, 95% bayesian block bootstrap over continuous trajectories. $p_\mathrm{cryo}$: 95% bayesian bootstrap over the continuous trajectories and the two half maps.

## Prerequisites

Stages 5–6 of `PIPELINE.md`:

- `MG_motifs.parquet` in each replica folder, `bias.COLVAR.<i>` in each replex replica folder (as for 03)
- `structures/7ez0/MG_motifs.parquet`, `structures/9cbu/MG_motifs.parquet`, `structures/<sys>/<ff>/repNN/MG_motifs_equil.parquet`: motifs of the deposited and equilibrated structures (`ion_motif_analysis.sh`)
- `density/blocks/{global,local/align10_out10}/<sys>_<ff>`: block means (`density/block_maps.py`)
- `density/motifs/{global,local/align10_out10}/<sys>_<ff>`: per-motif maps (`density/motif_maps.py`)
- `density/emd_42499_Con2-2.2A_half_{A,B}.mrc`, `structures/9CBU_fixed.pdb`
- for the highlighted motifs: replex trajectories (`run.xtc`, `input.gro`)

## Outputs

In `figures/`: `pcryo_local_global`, `nMg_mMg_cryo_{global,local}_agreement`, `mMg_REx_mMg_MD_cryo_{global,local}_agreement`, `psim_nMg_MD_mMg_REx_<sys>`, `psim_pcryo_<sim>`, `ccmax_pcryo`, `motif_analysis` (`.pdf`); `motif_table.tex`, `motif_count.tex`, `motif_recovery.tex`; `motif_highlight/<motif>/` (`{present,absent,best_mixture,ref_map}.mrc`, `{present,absent}.pdb`, `<motif>_best_f.pdf`) and `motif_highlight/legend.pdf`.

In [ ]:
import itertools
import os
import warnings

import matplotlib.pyplot as plt
import MDAnalysis as mda
import numpy as np
import pandas as pd
from cryomap import Grid, GridEnsemble
from labels import cm, simcolors, simlabels
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.markers import MarkerStyle
from MDAnalysis.analysis.align import alignto
from scipy.stats import linregress

from mg_sampling.motifs import compress_motifs, summarize_motifs
from mg_sampling.structure import standardize_names, unwrap_sequential, wrap_around_atomgroup
from pipeline import PROJECT_DIR
from pipeline.data import load_occupancy
from pipeline.density import (
    CORE_SEL,
    FAMILIES,
    TEMPFACTORS,
    best_tempfactor,
    half_maps,
    load_blocks,
    load_motif_maps,
    masks,
    mixture_cc_bootstrap,
    motif_coords,
    motif_maps_path,
    motif_values,
    motif_voxels,
    reference_rna,
    tempfactor_scan,
)
from pipeline.systems import SYSTEMS, replica_dirs

plt.style.use("paper.mplstyle")
os.makedirs("figures/motif_highlight", exist_ok=True)

## Motif occupancy in the simulations and structures

In [ ]:
# Mean occupancy p of each motif in each simulation (production window, λ0) and structure
MG_SYSTEMS = [s for s in SYSTEMS if 'nomg' not in s]
summaries = {}
for system in MG_SYSTEMS:
    occupancy = [b for k in range(SYSTEMS[system].n_replicates) for b in load_occupancy(system, k)]
    summaries[system.replace('/', '_')] = summarize_motifs(occupancy)


def structure_summary(files):
    # Fraction of the structures where each motif is occupied (no interval)
    summary = summarize_motifs([compress_motifs(pd.read_parquet(f)) for f in files])
    return summary.assign(p_low=np.nan, p_high=np.nan)


summaries['7ez0'] = structure_summary([PROJECT_DIR / 'structures/7ez0/MG_motifs.parquet'])
summaries['9cbu'] = structure_summary([PROJECT_DIR / 'structures/9cbu/MG_motifs.parquet'])
for sys_, ff in itertools.product(['7ez0', '7ez0_noion'], ['mMg_tip3p', 'nMg_tip3p']):
    files = sorted((PROJECT_DIR / 'structures' / sys_ / ff).glob('rep*/MG_motifs_equil.parquet'))
    summaries[f'equil_{sys_}_{ff}'] = structure_summary(files)

motif_summary = pd.concat(summaries, names=['sim']).reset_index()
motif_summary = motif_summary[motif_summary.groupby('motif')['p'].transform('max') > 0.05]
motif_info = motif_summary.drop_duplicates('motif').set_index('motif')
p_sim = motif_summary.set_index(['motif', 'sim'])[['p', 'p_low', 'p_high']]
print(len(motif_info), 'motifs with p > 0.05 somewhere')

In [ ]:
# Motif names for LaTeX tables ("text") and matplotlib ("mathrm")
MGRNA_NAMES = {
    'Oph': ('O', 'ph'), 'Ob': ('O', 'b'), 'Or': ('O', 'r'), 'Nb': ('N', 'b'),
}
MGRNA_STEREO = {
    'text': {'cis': r'\textit{cis}', 'trans': r'\textit{trans}', 'fac': r'\textit{fac}', 'mer': r'\textit{mer}',
             'Cis': r'\underline{Cis}', 'Trans': r'\underline{Trans}'},
    'mathrm': {'cis': r'$\mathit{cis}$', 'trans': r'$\mathit{trans}$', 'fac': r'$\mathit{fac}$',
               'mer': r'$\mathit{mer}$', 'Cis': r'$\mathrm{Cis}$', 'Trans': r'$\mathrm{Trans}$'},
}


def motif_label(x, style):
    mgrna = x['mgrna']
    for k, (a, b) in MGRNA_NAMES.items():
        mgrna = mgrna.replace(k, rf'$\{style}{{{a}}}_\{style}{{{b}}}$')
    for k, v in MGRNA_STEREO[style].items():
        mgrna = mgrna.replace(k, v)
    motif = ','.join(
        rf'$\{style}{{{resname}}}^\{style}{{{resid}}}_\{style}{{{name}}}$'
        for resname, resid, name in zip(x['inner_resnames'], x['inner_resids'], x['inner_names'])
    )
    return f'{motif}-{mgrna}' if x['mgrna'] else motif


motif_latex = motif_info.apply(motif_label, style='text', axis=1)
motif_mathtex = motif_info.apply(motif_label, style='mathrm', axis=1)

## Occupancy from the cryo-EM map

In [ ]:
# Pools of simulations ("+ Cryo"), and best B-factor of each simulation and family for the
# core phosphate shell (as in 05)
POOLS = {
    'plain_md_7ez0_mMg_tip3p_cryo': ['plain_md/7ez0/mMg_tip3p', 'plain_md/7ez0_noion/mMg_tip3p'],
    'plain_md_7ez0_nMg_tip3p_cryo': ['plain_md/7ez0/nMg_tip3p', 'plain_md/7ez0_noion/nMg_tip3p'],
    'replex_7ez0_mMg_tip3p_cryo': ['replex/7ez0/mMg_tip3p', 'replex/7ez0_noion/mMg_tip3p'],
}
ALIGNMENTS = ['local', 'global']
FRACTIONS = np.linspace(0, 1, 101)

halves = half_maps()
ref_map = 0.5 * (halves[0] + halves[1])
shell = masks(ref_map.params)['Core Phosphates (Shell only)']
blocks, tempfactor = {}, {}
for family, pool in itertools.product(FAMILIES, POOLS.values()):
    for system in pool:
        if not motif_maps_path(family, system).exists():
            print(f"No motif maps for {family} {system} (density/motif_maps.py)")
            continue
        blocks[family, system] = load_blocks(family, system)
        cc = tempfactor_scan(blocks[family, system].mean(), {'half': ref_map}, {'shell': shell})['half', 'shell']
        tempfactor[family, system] = best_tempfactor(TEMPFACTORS, cc)
pd.Series(tempfactor).unstack(0).round(1)

In [ ]:
def motif_blocks(family, system, motif, maps, voxels, coords):
    # Blurred maps (n_blocks, n_voxels) where `motif` is occupied (P) and not (A) in each
    # block of `system`, with their frame counts; A = (n·M - n_P·P) / (n - n_P), M the block means
    b = blocks[family, system]
    means = np.asarray(b.to_sparse(voxel_indices=voxels).data, dtype=np.float64)
    n = b.frame_info['n_frames']
    present, n_present = np.zeros_like(means), np.zeros(len(n))
    if motif in maps:
        ens = maps[motif]
        key = pd.MultiIndex.from_arrays([b.frame_info['replicate'], b.frame_info['replica']])
        rows = key.get_indexer(pd.MultiIndex.from_arrays([ens.frame_info['replicate'], ens.frame_info['replica']]))
        if np.any(rows < 0) or np.any(n[rows] != ens.frame_info['n_frames']):
            raise ValueError(f"{family} {system} {motif}: blocks differ from the block maps")
        present[rows], n_present[rows] = ens.data, ens.frame_info['n_present']
    n_absent = n - n_present
    absent = np.where(n_absent[:, None] > 0, (n[:, None] * means - n_present[:, None] * present) / np.maximum(n_absent, 1)[:, None], 0)
    both = GridEnsemble(np.concatenate([present, absent]), b.params.sparse(voxel_indices=voxels))
    values = np.asarray(motif_values(both, coords, tempfactor[family, system]).data)
    return values[:len(n)], n_present, values[len(n):], n_absent


rows = []
for family, (pool, systems) in itertools.product(FAMILIES, POOLS.items()):
    if not all((family, s) in blocks for s in systems):
        continue
    maps = {s: load_motif_maps(family, s, motif_info.index) for s in systems}
    for seed, (motif, info) in enumerate(motif_info.iterrows()):
        coords = motif_coords(info)
        if coords is None:  # ligand atoms missing from 9CBU
            continue
        voxels = motif_voxels(blocks[family, systems[0]].params, coords)
        parts = [motif_blocks(family, s, motif, maps[s], voxels, coords) for s in systems]
        present, n_present, absent, n_absent = (np.concatenate(x) for x in zip(*parts))
        if n_present.sum() == 0 or n_absent.sum() == 0:
            continue
        ref = motif_values(GridEnsemble.from_grids([h.to_sparse(voxel_indices=voxels) for h in halves]), coords)
        cc, best = mixture_cc_bootstrap(present, n_present, absent, n_absent, np.asarray(ref.data), FRACTIONS,
                                        n_resamples=1000, rng=seed)
        target = ref.params
        rows.append({
            'motif': motif, 'sim': pool, 'alignment': family,
            'cc': cc.estimate, 'cc_mean': cc.bootstrap_distribution.mean(axis=0),
            'cc_low': cc.confidence_interval.low, 'cc_high': cc.confidence_interval.high,
            'cc_max': cc.estimate.max(), 'best_f': best.estimate, 'best_f_mean': best.bootstrap_distribution.mean(),
            'best_f_low': best.confidence_interval.low, 'best_f_high': best.confidence_interval.high,
            'map_present': Grid(np.average(present, axis=0, weights=n_present), target),
            'map_absent': Grid(np.average(absent, axis=0, weights=n_absent), target),
            'map_ref': ref.mean(),
        })
result = pd.DataFrame(rows).set_index(['motif', 'sim', 'alignment']).sort_index()
result[['cc_max', 'best_f', 'best_f_low', 'best_f_high']].groupby(['sim', 'alignment']).describe().T.round(2)

In [ ]:
def ccc(x, y):
    # Concordance correlation coefficient
    cov = np.cov(x, y)
    return 2 * cov[0, 1] / (cov[0, 0] + cov[1, 1] + (np.mean(x) - np.mean(y))**2)


def pcryo(sim, alignment, column='best_f_mean'):
    return result.xs((sim, alignment), level=('sim', 'alignment'))[column].dropna()


pools = [pool for pool in POOLS if pool in result.index.get_level_values('sim')]

How well do the local and global fits agree?

In [ ]:
fig, axarr = plt.subplots(ncols=len(pools), figsize=(17.8*cm, 6.5*cm), squeeze=False)
for pool, ax in zip(pools, axarr[0]):
    x, y = pcryo(pool, 'local').align(pcryo(pool, 'global'), join='inner')
    ax.plot([0, 1], [0, 1], color='silver', ls='dashed', lw=1)
    ax.scatter(x, y, s=2)
    ax.set_aspect(1)
    ax.set_xlabel(r'$p_\mathrm{cryo}$ (local)')
    ax.set_ylabel(r'$p_\mathrm{cryo}$ (global)')
    ax.set_title(simlabels[pool])
    ax.text(0, 1, f"CCC = {ccc(x, y):.2f}", va='top')
fig.savefig("figures/pcryo_local_global.pdf")

How well do the force fields and methods agree?

In [ ]:
DENTICITY_COLORS = {1: 'silver', 2: 'royalblue', 3: 'crimson', 4: 'purple', 5: 'black'}
DENTICITY_MARKERS = {1: '.', 2: 'o', 3: '^', 4: 's', 5: '*'}
PAIRS = [  # (x, y, file name)
    ('plain_md_7ez0_nMg_tip3p_cryo', 'replex_7ez0_mMg_tip3p_cryo', 'nMg_mMg_cryo_{}_agreement'),
    ('plain_md_7ez0_nMg_tip3p_cryo', 'plain_md_7ez0_mMg_tip3p_cryo', 'mMg_REx_mMg_MD_cryo_{}_agreement'),
]
for (xsim, ysim, name), alignment in itertools.product(PAIRS, ALIGNMENTS):
    if xsim not in pools or ysim not in pools:
        continue
    fig, ax = plt.subplots(figsize=(8.7*cm, 6.5*cm))
    x, y = pcryo(xsim, alignment).align(pcryo(ysim, alignment), join='inner')
    denticity = motif_info.loc[x.index, 'n_inner']
    ax.plot([0, 1], [0, 1], color='silver', ls='dashed', lw=1, zorder=-1)
    for n_inner in np.sort(denticity.unique()):
        sel = denticity == n_inner
        ax.scatter(x[sel], y[sel], s=6, color=DENTICITY_COLORS[n_inner], marker=DENTICITY_MARKERS[n_inner], label=n_inner)
    ax.set_aspect(1)
    ax.set_xlabel(r'$p_\mathrm{cryo}$ ' + simlabels[xsim.removesuffix('_cryo')])
    ax.set_ylabel(r'$p_\mathrm{cryo}$ ' + simlabels[ysim.removesuffix('_cryo')])
    ax.set_title(alignment.capitalize() + ' alignment')
    ax.text(0, 1, f'CCC = {ccc(x, y):.2f}', va='top')
    ax.text(1, 0, f'$N = {len(x)}$', va='bottom', ha='right')
    ax.legend(title='Denticity', bbox_to_anchor=(1, 1))
    fig.savefig(f'figures/{name.format(alignment)}.pdf')

In [ ]:
# Occupancy in the simulations: nMg plain MD vs μMg replica exchange
for sys_ in ['7ez0', '7ez0_noion']:
    xsim, ysim = f'plain_md_{sys_}_nMg_tip3p', f'replex_{sys_}_mMg_tip3p'
    x, y = p_sim.xs(xsim, level='sim')['p'].align(p_sim.xs(ysim, level='sim')['p'], join='inner')
    fig, ax = plt.subplots(figsize=(8.7*cm, 6.5*cm))
    ax.plot([0, 1], [0, 1], color='silver', ls='dashed', lw=1)
    ax.scatter(x, y, s=2)
    ax.set_aspect(1)
    ax.set_xlabel(r'$p_\mathrm{sim}$ ' + simlabels[xsim])
    ax.set_ylabel(r'$p_\mathrm{sim}$ ' + simlabels[ysim])
    ax.text(0, 1, f'CCC = {ccc(x, y):.2f}', va='top')
    ax.text(1, 0, f'$N = {len(x)}$', va='bottom', ha='right')
    fig.savefig(f'figures/psim_nMg_MD_mMg_REx_{sys_}.pdf')

How well do the simulations and the cryo-EM map agree?

In [ ]:
alignment = 'local'
for pool, sys_ in itertools.product(['replex_7ez0_mMg_tip3p_cryo', 'plain_md_7ez0_nMg_tip3p_cryo'], ['7ez0', '7ez0_noion']):
    if pool not in pools:
        continue
    sim = pool.removesuffix('_cryo').replace('7ez0', sys_)
    sim_data = p_sim.xs(sim, level='sim')
    cryo = result.xs((pool, alignment), level=('sim', 'alignment'))
    index = sim_data.index.intersection(cryo['best_f_mean'].dropna().index)
    x, x_low, x_high = (sim_data.loc[index, c] for c in ('p', 'p_low', 'p_high'))
    y, y_low, y_high = (cryo.loc[index, c] for c in ('best_f_mean', 'best_f_low', 'best_f_high'))
    over = (x_low > y) & (y_high < x)
    under = (x_high < y) & (y_low > x)
    fig, ax = plt.subplots(figsize=(8.7*cm, 6.5*cm))
    ax.plot([0, 1], [0, 1], color='silver', ls='dashed', lw=1)
    for sel, color, what in [(over, 'crimson', 'Overpredicted'), (under, 'royalblue', 'Underpredicted'),
                             (~(over | under), 'silver', 'Correctly predicted')]:
        ax.errorbar(x[sel], y[sel], xerr=[(x - x_low).clip(0)[sel], (x_high - x).clip(0)[sel]],
                    yerr=[(y - y_low).clip(0)[sel], (y_high - y).clip(0)[sel]],
                    fmt='.', elinewidth=1, markersize=5, color=color)
        print(simlabels[sim], f"{what} by simulation: {sel.sum()}/{len(sel)}")
    ax.set_aspect(1)
    ax.set_xlabel(r'$p_\mathrm{sim}$')
    ax.set_ylabel(r'$p_\mathrm{cryo}$')
    ax.set_title(simlabels[sim] + f' ({alignment})')
    ax.text(0, 1, f'$CCC = {ccc(x, y):.2f}$', va='top')
    ax.text(1, 0, f'$N = {len(x)}$', va='bottom', ha='right')
    fig.savefig(f'figures/psim_pcryo_{sim}.pdf')

In [ ]:
# p_cryo vs the best correlation, for motifs in the core and in the peripheral helices
core_resids = set(reference_rna().select_atoms(CORE_SEL).resids)
is_core = motif_info['inner_resids'].map(lambda resids: set(resids) <= core_resids)

fig, axarr = plt.subplots(len(ALIGNMENTS), len(pools), figsize=(17.8*cm, 12*cm), squeeze=False)
for (i, alignment), (j, pool) in itertools.product(enumerate(ALIGNMENTS), enumerate(pools)):
    ax = axarr[i, j]
    data = result.xs((pool, alignment), level=('sim', 'alignment'))
    core = is_core.loc[data.index]
    ax.scatter(data.loc[core, 'cc_max'], data.loc[core, 'best_f_mean'], s=1.5, marker='o', color='royalblue', label='Core')
    ax.scatter(data.loc[~core, 'cc_max'], data.loc[~core, 'best_f_mean'], s=1, marker='s', color='crimson', label='Non core')
    ax.set_aspect(1)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel(r"$\mathrm{CC}_\mathrm{max}$")
    ax.set_ylabel(r"$p_\mathrm{cryo}$")
    ax.set_title(simlabels[pool.removesuffix('_cryo')] + f" {alignment.capitalize()}")
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc="outside lower center", ncols=2)
fig.savefig("figures/ccmax_pcryo.pdf")

## Motif table

In [ ]:
# Occupancy of the motifs in the structures and simulations, and from the cryo-EM map
# (global fit; not shown when the best correlation is below CC_MIN)
REFS = ['7ez0', '9cbu']
SIMS = ['equil_7ez0_mMg_tip3p', 'equil_7ez0_noion_mMg_tip3p', 'equil_7ez0_nMg_tip3p', 'equil_7ez0_noion_nMg_tip3p',
        'plain_md_7ez0_mMg_tip3p', 'plain_md_7ez0_nMg_tip3p', 'plain_md_7ez0_nMg_tip3p_soft', 'replex_7ez0_mMg_tip3p',
        'plain_md_7ez0_noion_mMg_tip3p', 'plain_md_7ez0_noion_nMg_tip3p', 'replex_7ez0_noion_mMg_tip3p']
CC_MIN = 0.7
alignment = 'global'

p = p_sim['p'].unstack('sim').reindex(columns=REFS + SIMS)
p_low = p_sim['p_low'].unstack('sim').reindex(columns=REFS + SIMS)
p_high = p_sim['p_high'].unstack('sim').reindex(columns=REFS + SIMS)
for pool in pools:
    cryo = result.xs((pool, alignment), level=('sim', 'alignment'))
    cryo = cryo[cryo['cc_max'] >= CC_MIN]
    p[pool], p_low[pool], p_high[pool] = (cryo[c].reindex(p.index) for c in ('best_f', 'best_f_low', 'best_f_high'))
sims = SIMS + pools

average = p.fillna(0).mean(axis=1)
order = pd.DataFrame({'7ez0': p['7ez0'].fillna(0), '9cbu': p['9cbu'].fillna(0), 'avg': average}).sort_values(
    ['7ez0', '9cbu', 'avg'], ascending=False).index
keep = (average >= 0.05) | p['7ez0'].notna() | p['9cbu'].notna()
order = order[keep.loc[order]]
p, p_low, p_high = (x.loc[order].reset_index() for x in (p, p_low, p_high))
p.index += 1
p_low.index += 1
p_high.index += 1
p

In [ ]:
def mg_labels(sim, pdb):
    # Mg²⁺ of each motif of a structure, as name^resid
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message=".*CRYST1 record")
        u = mda.Universe(pdb)
    data = motif_summary.query("sim == @sim").set_index('motif')['mg_indices']
    return data.map(lambda x: f"$\\text{{{u.atoms[x[0]].name}}}^\\text{{{u.atoms[x[0]].resid}}}$")


values = p[sims]
text = (values.map("{:.1f}".format) + " (" + p_low[sims].map("{:.1f}".format) + "-"
        + p_high[sims].map("{:.1f}".format) + ")").replace("nan (nan-nan)", None)
motif_table = p[['motif']].join(text)
motif_table['7ez0'] = p['motif'].map(mg_labels('7ez0', PROJECT_DIR / 'structures/7EZ0_fixed.pdb'))
motif_table['9cbu'] = p['motif'].map(mg_labels('9cbu', PROJECT_DIR / 'structures/9cbu/9CBU_fixed.pdb')).fillna('')
motif_table = motif_table[['motif'] + REFS + sims]
motif_table['motif'] = motif_table['motif'].map(motif_latex)
motif_table = motif_table.rename(columns={'motif': 'Motif'}).reset_index(names='Id')

styler = motif_table.style.set_properties(**{'background-color': '#FEFEFE', 'color': '#000000'}).background_gradient(
    axis=None, cmap="Reds", gmap=values.reset_index(drop=True), subset=sims
).highlight_null(color="white").highlight_null(color="#AAAAAA", subset=pools).relabel_index(
    [simlabels.get(col, col) for col in motif_table.columns], axis=1
).format(na_rep='').hide(axis='index')

latex_code = styler.to_latex(
    column_format=f"p{{0.5cm}}p{{4.2cm}}p{{0.8cm}}p{{0.8cm}} *{{{len(motif_table.columns) - 4}}}{{X}}\\label{{stable:motif_table}}",
    hrules=True,
    convert_css=True,
)
latex_code = latex_code.replace(r'begin{tabular}', r'begin{xltabular}{\linewidth}').replace(r'end{tabular}', r'end{xltabular}')
with open('figures/motif_table.tex', 'w') as f:
    f.write(latex_code)
styler

In [ ]:
# Occupancy of the first motifs of the table (squares: lower and upper bounds)
n_motifs = 60
cols = REFS + [sim for sim in sims if not sim.startswith('equil')]
n_sims = len(cols)
n_cryo = len(pools)

fig, ax = plt.subplots(figsize=(17.8*cm, 5*cm))
ax.grid(None)
y_range = np.arange(n_sims).astype(float)
y_range[n_sims - n_cryo:] += 0.5
y_range[:len(REFS)] -= 0.5
low = p_low[cols].iloc[:n_motifs].copy()
high = p_high[cols].iloc[:n_motifs].copy()
for ref in REFS:  # structures: present or not
    low[ref] = high[ref] = p[ref].iloc[:n_motifs].where(p[ref].iloc[:n_motifs] > 0)
for i in range(len(low)):
    ax.scatter(1 + np.full(n_sims, i), y_range, c=low.iloc[i], cmap='gnuplot2_r', edgecolors='k', s=20,
               marker=MarkerStyle('s', fillstyle='bottom'), vmin=0, vmax=1, clip_on=False, linewidths=0.5)
    ax.scatter(1 + np.full(n_cryo, i), y_range[n_sims - n_cryo:], marker='x', facecolor='silver', lw=1, s=10, zorder=-10)
    ax.scatter(1 + np.full(n_sims, i), y_range, c=high.iloc[i], cmap='gnuplot2_r', edgecolors='k', s=20,
               marker=MarkerStyle('s', fillstyle='top'), vmin=0, vmax=1, clip_on=False, linewidths=0.5)
ax.tick_params(top=True, labeltop=True, bottom=True, labelbottom=True)
ax.spines[['left', 'right']].set_visible(False)
ax.set_yticks(y_range, [simlabels.get(col, col) for col in cols])
ax.set_xlabel('Motif index')
ax.set_xlim(0.5, n_motifs + 0.5)
ax.set_ylim(y_range[-1] + 0.5, y_range[0] - 0.5)
cbar = fig.colorbar(ScalarMappable(norm=Normalize(0, 1), cmap='gnuplot2_r'), ax=ax, pad=0.02, fraction=0.01, label='Frequency')
cbar.ax.set_yticks([0, 1])
fig.savefig('figures/motif_analysis.pdf')

In [ ]:
# Number of motifs with p > 0.05, by denticity
x = motif_summary.query('p > 0.05').groupby(['sim', 'n_inner'])['p'].count().unstack().fillna(0).astype(int).rename(index=simlabels)
x = pd.concat([x, x.sum(axis=1).rename('')], keys=(r'Denticity', 'Tot'), axis=1)
x.index.name = None
x.columns.name = r'$n_\mathrm{inner}$'
styler = x.style.background_gradient(cmap='Reds')
with open('figures/motif_count.tex', 'w') as f:
    f.write(styler.to_latex(column_format=f"l{'r' * len(x.columns)}", hrules=True, convert_css=True))
styler

In [ ]:
# Number of motifs of the structures recovered (p > 0.05) by each simulation
motif_7ez0 = motif_summary.query('sim == "7ez0"')['motif']
motif_9cbu = motif_summary.query('sim == "9cbu"')['motif']
x = motif_summary.query('p > 0.05').groupby(['sim', 'n_inner']).apply(
    lambda x: pd.Series({'7ez0': np.isin(x['motif'], motif_7ez0).sum(), '9cbu': np.isin(x['motif'], motif_9cbu).sum()})
)
x = x.unstack().fillna(0).astype(int)
x = x.loc[:, ~(x == 0).all()]
x.index.name = None
xsum = x.T.groupby(level=0).sum().T
xsum.columns = pd.MultiIndex.from_product([xsum.columns, ['Tot']])
x = x.join(xsum).sort_index(axis=1).rename(simlabels)
x['7ez0'] = x['7ez0'].astype(str) + '/' + x['7ez0'].loc['7ez0'].astype(str)
x['9cbu'] = x['9cbu'].astype(str) + '/' + x['9cbu'].loc['9cbu'].astype(str)
styler = x.style.background_gradient(cmap='Reds', gmap=x.map(lambda v: int(v.split('/')[0])), axis=None)
with open('figures/motif_recovery.tex', 'w') as f:
    f.write(styler.to_latex(column_format=f"l{'r' * len(x.columns)}", hrules=True, convert_css=True))
styler

## Highlighted motifs

In [ ]:
HIGHLIGHT = [
    "G257OP1,U258OP2,C260O2,U305OP2-Cis-3OphOb",
    "G257OP1,U258OP2,U305OP2-mer-3Oph",
    "A184OP1,A186OP1,A187OP2-mer-3Oph",
    "A184OP1,A186OP1,A187OP2,G188OP2-Trans-4Oph",
]
HIGHLIGHT_POOL = 'replex_7ez0_mMg_tip3p_cryo'
HIGHLIGHT_SYSTEM = 'replex/7ez0/mMg_tip3p'
alignment = 'global'


def scaled(grid, ref):
    # Linear fit of `grid` onto `ref` (same voxels)
    fit = linregress(np.asarray(grid.data), np.asarray(ref.data))
    return fit.slope * grid + fit.intercept


def example_frame(motif, occupied):
    # (replica folder, replica index, time in ps) of the first production frame at λ0 where
    # `motif` is occupied (or not)
    for k in range(SYSTEMS[HIGHLIGHT_SYSTEM].n_replicates):
        for i, ((occupancy, _), path) in enumerate(zip(load_occupancy(HIGHLIGHT_SYSTEM, k), replica_dirs(HIGHLIGHT_SYSTEM, k))):
            counts = occupancy[motif] if motif in occupancy else pd.Series(0, index=occupancy.index)
            times = counts.index[(counts > 0) == occupied]
            if len(times):
                return path, i, times[0]


ref_u = mda.Universe(PROJECT_DIR / "structures/9CBU_fixed.pdb")
for motif in HIGHLIGHT:
    if (motif, HIGHLIGHT_POOL, alignment) not in result.index:
        print(f"{motif}: no result for {HIGHLIGHT_POOL} ({alignment})")
        continue
    data = result.loc[motif, HIGHLIGHT_POOL, alignment]
    out = f'figures/motif_highlight/{motif}'
    os.makedirs(out, exist_ok=True)
    present, absent = scaled(data['map_present'], data['map_ref']), scaled(data['map_absent'], data['map_ref'])
    for name, grid in [('present', present), ('absent', absent), ('ref_map', data['map_ref']),
                       ('best_mixture', data['best_f'] * present + (1 - data['best_f']) * absent)]:
        grid.to_dense().write_mrc(f'{out}/{name}.mrc')

    # One frame with and one without the motif, fitted on the RNA atoms within 10 Å of it
    info = motif_info.loc[motif]
    for what, occupied in [('present', True), ('absent', False)]:
        path, i, t = example_frame(motif, occupied)
        u = mda.Universe(path / 'input.gro', path / 'run.xtc')
        standardize_names(u)
        u.trajectory[int(round(t / u.trajectory.dt))]
        rna = u.select_atoms('nucleic and not type H')
        unwrap_sequential(rna)
        wrap_around_atomgroup(u.atoms, rna.select_atoms("name P"))
        motif_sel = u.select_atoms(*(f"resname {rn} and resid {ri} and name {n}" for rn, ri, n in
                                     zip(info['inner_resnames'], info['inner_resids'], info['inner_names'])))
        mobile = rna.select_atoms("group motif_sel or around 10 group motif_sel", motif_sel=motif_sel)
        ref = ref_u.select_atoms(*(f"resname {a.resname} and resid {a.resid} and name {a.name}" for a in mobile))
        assert len(mobile) == len(ref)
        alignto(mobile, ref)
        u.atoms.write(f'{out}/{what}.pdb')

In [ ]:
# Correlation with the cryo-EM map against the occupancy of the motif in the mixture
for motif in HIGHLIGHT:
    fig, ax = plt.subplots(figsize=(4.5*cm, 4.5*cm))
    for i, pool in enumerate(pools):
        if (motif, pool, alignment) not in result.index:
            continue
        data = result.loc[motif, pool, alignment]
        ax.plot(FRACTIONS, data['cc'], color=simcolors[pool], label=simlabels[pool])
        ax.fill_between(FRACTIONS, data['cc_low'], data['cc_high'], color=simcolors[pool], alpha=0.3)
        ax.axvline(data['best_f_mean'], color=simcolors[pool], ls=(i, (3, 3)))
        ax.axvspan(data['best_f_low'], data['best_f_high'], color=simcolors[pool], alpha=0.3)
    ax.set_xlim(-0.01, 1.01)
    ax.set_ylabel("Pearson correlation")
    ax.set_xlabel("Motif occupancy")
    fig.suptitle(motif_mathtex.loc[motif], fontsize=8)
    os.makedirs(f"figures/motif_highlight/{motif}", exist_ok=True)
    fig.savefig(f"figures/motif_highlight/{motif}/{motif}_best_f.pdf")

handles, labels = ax.get_legend_handles_labels()
fig_legend, ax_legend = plt.subplots(figsize=(17.8*cm, 1*cm))
ax_legend.axis('off')
ax_legend.legend(handles, labels, ncols=len(labels), loc='center')
fig_legend.savefig("figures/motif_highlight/legend.pdf")